# NOMAD RAG — Gold Q&A Reviewer (v3)

A cleaner UI for curating candidate Q&As into your gold set.

**Improvements over v2**
- Two-column layout (selector on the left, preview on the right) — no overlap.
- Preview panel scrolls independently (won’t block the selector).
- Adjustable preview length + "Show full answers" toggle.
- Clear success banner with appended items.
- Gold summary refreshes automatically.

> Expected input: `eval/gold_review.csv` with columns: `question, proposed_answer, source_url, title, section, method, score, id`.


In [2]:
import os, json, re
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML
import ipywidgets as W

# ---- Paths (edit if needed) ----
CANDIDATE_CSV = Path("../eval/gold_review.csv")
GOLD_JSONL    = Path("../eval/gold_nomad.jsonl")
GOLD_JSONL.parent.mkdir(parents=True, exist_ok=True)

if not CANDIDATE_CSV.exists():
    raise FileNotFoundError(f"Candidate CSV not found: {CANDIDATE_CSV.resolve()}")

df_all = pd.read_csv(CANDIDATE_CSV)

required = {"question","proposed_answer","source_url","title","section","method","score","id"}
missing = required - set(df_all.columns)
if missing:
    raise ValueError(f"Candidate CSV missing columns: {missing}")

def load_gold(path: Path):
    if not path.exists(): return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

gold_list = load_gold(GOLD_JSONL)

def norm_question(q: str) -> str:
    q = (q or "").lower().strip()
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"[^\w\s\?\-]", "", q)  # keep ? and -
    return q

existing_keys = {(norm_question(r.get("question","")), r.get("source_url","")) for r in gold_list}

print(f"Loaded {len(df_all)} candidates")
print(f"Loaded {len(gold_list)} existing gold entries from {GOLD_JSONL}")


Loaded 69 candidates
Loaded 0 existing gold entries from ../eval/gold_nomad.jsonl


In [4]:
# --- LOCKED LEFT PANE UI (replace your layout/widgets cell with this) ---
import ipywidgets as W
from IPython.display import display, Markdown, HTML
import re, json, pandas as pd

# Filters
min_score = W.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.05,
                          description="Min score", continuous_update=False,
                          layout=W.Layout(width="320px"))
method_opts = ["(any)"] + sorted(df_all["method"].dropna().unique().tolist())
method_dd = W.Dropdown(options=method_opts, value="(any)", description="Method",
                       layout=W.Layout(width="260px"))
search_box = W.Text(value="", description="Search",
                    placeholder="text in question/answer/title/section",
                    layout=W.Layout(width="360px"))
refresh_btn = W.Button(description="Refresh", layout=W.Layout(width="120px"))
top_bar = W.HBox([min_score, method_dd, search_box, refresh_btn])

# LEFT: selector column (LOCKED width + its own scroll)
select_box = W.SelectMultiple(
    options=[], rows=24, description="Select",
    layout=W.Layout(width="520px", height="560px", overflow="auto")
)
add_btn = W.Button(description="Add to Gold", button_style="success",
                   layout=W.Layout(width="160px"))
status_out = W.Output(layout=W.Layout(width="520px"))

left_col = W.VBox(
    [select_box, W.HBox([add_btn]), status_out],
    layout=W.Layout(
        width="540px", min_width="540px", max_width="540px",
        overflow="hidden"  # left column itself doesn't scroll horizontally
    )
)

# RIGHT: preview column (independent scroll)
preview_len = W.IntSlider(value=1200, min=200, max=6000, step=100,
                          description="Preview chars", readout=True,
                          layout=W.Layout(width="360px"))
full_toggle = W.Checkbox(value=False, description="Show full answers")
preview_header = W.HBox([preview_len, full_toggle])

preview_out = W.Output(
    layout=W.Layout(border="1px solid #ddd", padding="8px",
                    height="560px", overflow="auto", width="100%")
)
right_col = W.VBox(
    [preview_header, preview_out],
    layout=W.Layout(min_width="600px", overflow="hidden")
)

# Use a GRID (prevents overlap)
main_area = W.GridBox(
    children=[left_col, right_col],
    layout=W.Layout(
        grid_template_columns="540px 1fr",   # left fixed, right flexible
        grid_gap="12px",
        width="100%"
    )
)

gold_summary_out = W.Output()

display(top_bar, main_area, W.HTML("<hr/>"), gold_summary_out)

def format_option(row):
    q_short = (row["question"][:96] + "…") if len(row["question"]) > 100 else row["question"]
    return f"[{row['score']:.2f}] {row['method']}: {q_short}  —  ({row['title']} › {row['section']})"

def filtered_df():
    df = df_all.copy()
    df = df[df["score"] >= float(min_score.value)]
    if method_dd.value != "(any)":
        df = df[df["method"] == method_dd.value]
    q = search_box.value.strip().lower()
    if q:
        mask = (
            df["question"].str.lower().str.contains(q, na=False) |
            df["proposed_answer"].str.lower().str.contains(q, na=False) |
            df["title"].str.lower().str.contains(q, na=False) |
            df["section"].str.lower().str.contains(q, na=False)
        )
        df = df[mask]
    return df

def apply_filter(_=None):
    df = filtered_df()
    select_box.options = [(format_option(row), row["id"]) for _, row in df.iterrows()]
    with status_out:
        status_out.clear_output()
        print(f"Filtered candidates: {len(df)}")
    update_preview()

def shorten(text: str, limit: int) -> str:
    return text if len(text) <= limit else text[:limit] + " …"

def update_preview(_=None):
    df = filtered_df()
    idx = {row["id"]: row for _, row in df.iterrows()}
    chosen = list(select_box.value)[:8]
    with preview_out:
        preview_out.clear_output()
        if not chosen:
            display(Markdown("> Select candidates on the left to preview…"))
            return
        for cid in chosen:
            row = idx.get(cid)
            if row is None:
                continue
            ans = row["proposed_answer"] if full_toggle.value else shorten(row["proposed_answer"], int(preview_len.value))
            html = f"""
            <div style='border:1px solid #eee; padding:10px; margin:8px 0; border-radius:8px'>
              <div style='font-weight:600'>Question:</div>
              <div style='margin:4px 0 8px 0'>{row['question']}</div>
              <div style='font-weight:600'>Proposed answer:</div>
              <div style='margin:4px 0 8px 0'>{ans}</div>
              <div style='margin-top:6px'>
                <b>Source</b>: <a href='{row['source_url']}' target='_blank'>{row['title']} › {row['section']}</a>
                &nbsp; | &nbsp; <i>{row['method']}</i> &nbsp; score={row['score']:.2f}
              </div>
            </div>
            """
            display(HTML(html))

def show_gold_summary():
    if GOLD_JSONL.exists():
        with GOLD_JSONL.open("r", encoding="utf-8") as f:
            gold = [json.loads(line) for line in f if line.strip()]
    else:
        gold = []
    with gold_summary_out:
        gold_summary_out.clear_output()
        display(Markdown(f"**Gold entries:** {len(gold)}"))
        if gold:
            gdf = pd.DataFrame(gold)
            display(gdf[["question","gold_urls"]].head(10))

def add_selected_to_gold(_):
    df = filtered_df()
    by_id = {row["id"]: row for _, row in df.iterrows()}
    picked_ids = list(select_box.value)
    if not picked_ids:
        with status_out:
            status_out.clear_output()
            display(Markdown("⚠️ **No candidates selected.**"))
        return

    added, new_recs = [], []
    for cid in picked_ids:
        row = by_id.get(cid)
        if row is None:
            continue
        key = (norm_question(row["question"]), row["source_url"])
        if key in existing_keys:
            continue
        rec = {
            "question": row["question"],
            "gold_answer": row["proposed_answer"],
            "gold_urls": [row["source_url"]],
            "title": row.get("title",""),
            "section": row.get("section",""),
            "source_url": row["source_url"],
            "meta": {"method": row.get("method",""), "score": float(row.get("score",0)), "id": row.get("id","")}
        }
        new_recs.append(rec)
        added.append(row["question"])
        existing_keys.add(key)

    if new_recs:
        with GOLD_JSONL.open("a", encoding="utf-8") as f:
            for r in new_recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    with status_out:
        status_out.clear_output()
        if new_recs:
            items = "<br/>".join("• " + q for q in added[:6])
            more = "" if len(added) <= 6 else f"<br/>… and {len(added)-6} more"
            display(HTML(f"<div style='background:#e9f7ef;border-left:6px solid #2ecc71;padding:10px'>"
                         f"<b>Added {len(added)}</b> entries to <code>{GOLD_JSONL}</code>."
                         f"<div style='margin-top:6px'>{items}{more}</div></div>"))
        else:
            display(HTML("<div style='background:#fdecea;border-left:6px solid #e74c3c;padding:10px'>Nothing added (all duplicates).</div>"))
    show_gold_summary()

# wire events
refresh_btn.on_click(apply_filter)
select_box.observe(update_preview, names="value")
add_btn.on_click(add_selected_to_gold)
preview_len.observe(update_preview, names="value")
full_toggle.observe(update_preview, names="value")

# render
apply_filter()
show_gold_summary()


GridBox(children=(VBox(children=(SelectMultiple(description='Select', layout=Layout(height='560px', overflow='…

HTML(value='<hr/>')

Output()